<a href="https://colab.research.google.com/github/michael-borck/loco-llm/blob/main/notebooks/train_tiny_poc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LocoLLM: TinyLlama PoC — Train 3 Adapters in One Session

This notebook trains **all 3 LocoLLM adapters** (math, code, analysis) on **TinyLlama-1.1B** in a single Colab session. It’s a fast proof-of-concept so you can see the complete workflow before investing time in Qwen3-4B adapters.

**Why TinyLlama?** It’s small enough to train 3 adapters in ~20–30 minutes total on a free T4. The results won’t be great — and that’s the point. Seeing where a 1.1B model struggles motivates why base model selection matters.

**What you get:** 3 GGUF files (~600MB each) that you can load into Ollama and test locally.

**What you need:** A free Google Colab account with a T4 GPU runtime.

---

## How to use this notebook

1. Open in Colab (click the badge above)
2. Go to **Runtime > Change runtime type** and select **T4 GPU**
3. Run all cells in order (**Runtime > Run all**)
4. Download the 3 GGUF files from the final cells
5. Load them into Ollama locally (instructions at the end)

## Step 0: Verify GPU

Make sure you have a T4 (or better) GPU assigned.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## Step 1: Install Unsloth

This takes 2–3 minutes. Unsloth pulls in the right versions of transformers, peft, trl, etc.

In [ ]:
!pip install --upgrade --no-cache-dir unsloth unsloth_zoo 2>&1 | tail -5

## Step 2: Define Helper Functions

The load → LoRA → train → export cycle is the same for all 3 adapters. We define it once here, then call it 3 times with different datasets.

Each call loads TinyLlama fresh (takes ~10 seconds in 4-bit). This is necessary because LoRA weights are per-task — we need a clean base model each time.

In [ ]:
import os

from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastModel

# Shared config
MODEL_NAME = "unsloth/tinyllama-bnb-4bit"
MAX_SEQ_LENGTH = 1024
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
NUM_EPOCHS = 3
BATCH_SIZE = 2
GRADIENT_ACCUMULATION = 4
LEARNING_RATE = 2e-4
QUANTIZATION_METHOD = "q4_k_m"


def load_and_train_adapter(training_data, adapter_name):
    """Load TinyLlama, apply LoRA, train on data, return (model, tokenizer)."""
    print(f"\n{'='*60}")
    print(f"Training: {adapter_name}")
    print(f"{'='*60}")

    # Load fresh base model
    model, tokenizer = FastModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
        full_finetuning=False,
    )

    # Apply LoRA
    model = FastModel.get_peft_model(
        model,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

    # Prepare dataset
    dataset = Dataset.from_list(training_data)

    def format_conversations(examples):
        texts = []
        for convos in examples["conversations"]:
            text = tokenizer.apply_chat_template(
                convos, tokenize=False, add_generation_prompt=False,
            )
            texts.append(text)
        return {"text": texts}

    dataset = dataset.map(format_conversations, batched=True)
    print(f"Dataset ready: {len(dataset)} examples")

    # Train
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        args=TrainingArguments(
            output_dir=f"./checkpoints-{adapter_name}",
            per_device_train_batch_size=BATCH_SIZE,
            gradient_accumulation_steps=GRADIENT_ACCUMULATION,
            num_train_epochs=NUM_EPOCHS,
            learning_rate=LEARNING_RATE,
            lr_scheduler_type="cosine",
            warmup_ratio=0.1,
            fp16=True,
            logging_steps=5,
            save_strategy="epoch",
            seed=42,
            optim="adamw_8bit",
            report_to="none",
        ),
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        packing=False,
    )

    print(f"Training: {NUM_EPOCHS} epochs, effective batch size {BATCH_SIZE * GRADIENT_ACCUMULATION}")
    trainer.train()
    print(f"{adapter_name} training complete!")

    return model, tokenizer


def export_gguf(model, tokenizer, adapter_name):
    """Export merged GGUF for the trained adapter."""
    output_dir = f"./gguf-{adapter_name}"
    os.makedirs(output_dir, exist_ok=True)

    print(f"\nExporting {adapter_name} GGUF ({QUANTIZATION_METHOD})...")
    model.save_pretrained_gguf(
        output_dir,
        tokenizer,
        quantization_method=QUANTIZATION_METHOD,
    )

    for f in os.listdir(output_dir):
        if f.endswith(".gguf"):
            size_mb = os.path.getsize(os.path.join(output_dir, f)) / (1024 * 1024)
            print(f"Exported: {output_dir}/{f} ({size_mb:.0f} MB)")

    return output_dir


def sanity_check(model, tokenizer, questions):
    """Run a few test questions through the trained model."""
    FastModel.for_inference(model)
    for q in questions:
        messages = [{"role": "user", "content": q}]
        inputs = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_tensors="pt",
        ).to(model.device)
        outputs = model.generate(inputs, max_new_tokens=256, temperature=0.0, use_cache=True)
        response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
        print(f"Q: {q}")
        print(f"A: {response}")
        print("-" * 60)


print("Helper functions defined. Ready to train adapters!")

---

## Adapter 1: Math

### Download GSM8K Data

200 examples from [GSM8K](https://huggingface.co/datasets/openai/gsm8k) with step-by-step reasoning. Same data as the Qwen3 math notebook.

In [ ]:
import re

import requests

MATH_NUM_EXAMPLES = 200

url = f"https://datasets-server.huggingface.co/rows?dataset=openai/gsm8k&config=main&split=train&offset=0&length={MATH_NUM_EXAMPLES}"
resp = requests.get(url, timeout=60)
resp.raise_for_status()
math_rows = [r["row"] for r in resp.json()["rows"]]
print(f"Downloaded {len(math_rows)} examples from GSM8K")


def format_answer(answer_text: str) -> str:
    """Convert GSM8K answer format to clean step-by-step reasoning."""
    parts = answer_text.split("####")
    reasoning = re.sub(r"<<.*?>>", "", parts[0]).strip()
    final_answer = parts[1].strip() if len(parts) > 1 else ""
    return f"{reasoning}\nThe answer is {final_answer}"


math_data = [
    {
        "conversations": [
            {"role": "user", "content": ex["question"].strip()},
            {"role": "assistant", "content": format_answer(ex["answer"])},
        ]
    }
    for ex in math_rows
]

print(f"Formatted {len(math_data)} math training examples")
print(f"\nSample question: {math_data[0]['conversations'][0]['content'][:100]}...")
print(f"Sample answer:   {math_data[0]['conversations'][1]['content'][:100]}...")

### Train Math Adapter

~5 minutes on T4.

In [ ]:
math_model, math_tokenizer = load_and_train_adapter(math_data, "math")

### Math Sanity Check

Quick test — TinyLlama math won’t be amazing, but it should attempt step-by-step reasoning.

In [ ]:
sanity_check(math_model, math_tokenizer, [
    "What is 15 + 27?",
    "A store has 120 apples. They sell 45 in the morning and 30 in the afternoon. How many apples are left?",
    "If a shirt costs $40 and is on sale for 25% off, what is the sale price?",
])

### Export Math GGUF

In [ ]:
math_gguf_dir = export_gguf(math_model, math_tokenizer, "math")

# Free memory before loading next adapter
del math_model, math_tokenizer
import gc; gc.collect()
import torch; torch.cuda.empty_cache()
print("Memory cleared for next adapter.")

---

## Adapter 2: Code

### Download Code Data

300 examples from [python_code_instructions_18k_alpaca](https://huggingface.co/datasets/iamtarun/python_code_instructions_18k_alpaca). We filter out examples with very short outputs (< 20 chars).

In [ ]:
CODE_NUM_EXAMPLES = 300
MIN_OUTPUT_LENGTH = 20

url = f"https://datasets-server.huggingface.co/rows?dataset=iamtarun/python_code_instructions_18k_alpaca&config=default&split=train&offset=0&length={CODE_NUM_EXAMPLES + 100}"
resp = requests.get(url, timeout=60)
resp.raise_for_status()
code_rows = [r["row"] for r in resp.json()["rows"]]
print(f"Downloaded {len(code_rows)} raw examples from python_code_instructions_18k_alpaca")

code_data = []
for ex in code_rows:
    if len(code_data) >= CODE_NUM_EXAMPLES:
        break

    output = (ex.get("output") or "").strip()
    if len(output) < MIN_OUTPUT_LENGTH:
        continue

    instruction = (ex.get("instruction") or "").strip()
    extra_input = (ex.get("input") or "").strip()
    user_msg = f"{instruction}\n\n{extra_input}" if extra_input else instruction

    if not user_msg:
        continue

    code_data.append(
        {
            "conversations": [
                {"role": "user", "content": user_msg},
                {"role": "assistant", "content": output},
            ]
        }
    )

print(f"Formatted {len(code_data)} code training examples")
print(f"\nSample question: {code_data[0]['conversations'][0]['content'][:100]}...")
print(f"Sample answer:   {code_data[0]['conversations'][1]['content'][:100]}...")

### Train Code Adapter

~5 minutes on T4.

In [ ]:
code_model, code_tokenizer = load_and_train_adapter(code_data, "code")

### Code Sanity Check

TinyLlama code output will be rough, but it should attempt Python syntax.

In [ ]:
sanity_check(code_model, code_tokenizer, [
    "Write a Python function that checks if a number is prime.",
    "Write a Python function to reverse a string.",
    "Write a Python function that returns the factorial of a number.",
])

### Export Code GGUF

In [ ]:
code_gguf_dir = export_gguf(code_model, code_tokenizer, "code")

del code_model, code_tokenizer
import gc; gc.collect()
torch.cuda.empty_cache()
print("Memory cleared for next adapter.")

---

## Adapter 3: Analysis

### Download Analysis Data

300 examples from [allenai/sciq](https://huggingface.co/datasets/allenai/sciq) — science reading comprehension. Each example has a passage, a question, and a correct answer.

In [ ]:
ANALYSIS_NUM_EXAMPLES = 300

url = f"https://datasets-server.huggingface.co/rows?dataset=allenai/sciq&config=default&split=train&offset=0&length={ANALYSIS_NUM_EXAMPLES + 100}"
resp = requests.get(url, timeout=60)
resp.raise_for_status()
analysis_rows = [r["row"] for r in resp.json()["rows"]]
print(f"Downloaded {len(analysis_rows)} raw examples from allenai/sciq")

analysis_data = []
for ex in analysis_rows:
    if len(analysis_data) >= ANALYSIS_NUM_EXAMPLES:
        break

    support = (ex.get("support") or "").strip()
    question = (ex.get("question") or "").strip()
    correct_answer = (ex.get("correct_answer") or "").strip()

    if not support or not question or not correct_answer:
        continue

    user_msg = (
        f"Read the following passage and answer the question.\n\n"
        f"Passage: {support}\n\n"
        f"Question: {question}"
    )

    assistant_msg = (
        f"Based on the passage, the answer is: {correct_answer}\n\n"
        f"The passage states that {support[:200].rstrip('.')}. "
        f'This tells us that the answer to "{question}" is {correct_answer}.'
    )

    analysis_data.append(
        {
            "conversations": [
                {"role": "user", "content": user_msg},
                {"role": "assistant", "content": assistant_msg},
            ]
        }
    )

print(f"Formatted {len(analysis_data)} analysis training examples")
print(f"\nSample question: {analysis_data[0]['conversations'][0]['content'][:100]}...")
print(f"Sample answer:   {analysis_data[0]['conversations'][1]['content'][:100]}...")

### Train Analysis Adapter

~5 minutes on T4.

In [ ]:
analysis_model, analysis_tokenizer = load_and_train_adapter(analysis_data, "analysis")

### Analysis Sanity Check

TinyLlama should attempt passage-based reasoning, even if the quality is limited.

In [ ]:
sanity_check(analysis_model, analysis_tokenizer, [
    "Read the following passage and answer the question.\n\nPassage: Water boils at 100 degrees Celsius at standard atmospheric pressure. At higher altitudes, the atmospheric pressure is lower, so water boils at a lower temperature.\n\nQuestion: Why does water boil at a lower temperature at higher altitudes?",
    "Read the following passage and answer the question.\n\nPassage: Photosynthesis is the process by which plants convert sunlight, water, and carbon dioxide into glucose and oxygen. This process takes place primarily in the leaves of plants, within structures called chloroplasts.\n\nQuestion: Where does photosynthesis primarily take place?",
    "Read the following passage and answer the question.\n\nPassage: The mitochondria are often called the powerhouse of the cell because they generate most of the cell's supply of adenosine triphosphate (ATP), which is used as a source of chemical energy.\n\nQuestion: What molecule do mitochondria primarily produce?",
])

### Export Analysis GGUF

In [ ]:
analysis_gguf_dir = export_gguf(analysis_model, analysis_tokenizer, "analysis")

del analysis_model, analysis_tokenizer
import gc; gc.collect()
torch.cuda.empty_cache()
print("Memory cleared.")

---

## Save to Google Drive (Optional)

Save all 3 GGUFs to Google Drive so they persist after the Colab session ends.

In [ ]:
import shutil

from google.colab import drive

drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/locollm-adapters/tinyllama"
os.makedirs(DRIVE_DIR, exist_ok=True)

for adapter_name, gguf_dir in [("math", math_gguf_dir), ("code", code_gguf_dir), ("analysis", analysis_gguf_dir)]:
    for f in os.listdir(gguf_dir):
        if f.endswith(".gguf"):
            src = os.path.join(gguf_dir, f)
            dst = os.path.join(DRIVE_DIR, f"tinyllama-{adapter_name}.gguf")
            print(f"Copying {adapter_name} GGUF to Google Drive...")
            shutil.copy2(src, dst)
            print(f"  Saved to: {dst}")

## Download All GGUFs

Download all 3 GGUF files to your local machine.

In [ ]:
from google.colab import files

for adapter_name, gguf_dir in [("math", math_gguf_dir), ("code", code_gguf_dir), ("analysis", analysis_gguf_dir)]:
    for f in os.listdir(gguf_dir):
        if f.endswith(".gguf"):
            print(f"Downloading {adapter_name} GGUF...")
            files.download(os.path.join(gguf_dir, f))

---

## Next Steps: Using Your TinyLlama Adapters Locally

### 1. Create Ollama Models from the GGUFs

For each downloaded GGUF, create an Ollama model:

```bash
# Create a Modelfile for each adapter
echo 'FROM ./unsloth.Q4_K_M.gguf' > Modelfile-tinyllama-math
echo 'FROM ./unsloth.Q4_K_M.gguf' > Modelfile-tinyllama-code
echo 'FROM ./unsloth.Q4_K_M.gguf' > Modelfile-tinyllama-analysis

# Create the Ollama models (run from your Downloads folder, or wherever the GGUFs are)
ollama create tinyllama-math -f Modelfile-tinyllama-math
ollama create tinyllama-code -f Modelfile-tinyllama-code
ollama create tinyllama-analysis -f Modelfile-tinyllama-analysis
```

### 2. Chat with Your Models

```bash
# Test each adapter directly
ollama run tinyllama-math "What is 15 + 27?"
ollama run tinyllama-code "Write a function to check if a number is prime."
ollama run tinyllama-analysis "Read the following passage and answer the question. Passage: Water boils at 100C at sea level. Question: At what temperature does water boil at sea level?"
```

### 3. Compare with Qwen3-4B

The TinyLlama adapters will be noticeably weaker than Qwen3-4B adapters. Try the same questions on both to see the difference. This is why base model selection matters!

### 4. Train the "Real" Adapters

Once you understand the workflow, train production-quality adapters on Qwen3-4B using the dedicated notebooks:

- [train_math_adapter.ipynb](https://github.com/michael-borck/loco-llm/blob/main/notebooks/train_math_adapter.ipynb) — Math adapter on Qwen3-4B

The Qwen3-4B adapters integrate with the full LocoLLM system (`loco chat`, `loco route`, `loco eval`).

---

## Training Summary

| Adapter | Dataset | Examples | Base Model | GGUF Size |
|---------|---------|----------|------------|----------|
| Math | GSM8K | 200 | TinyLlama-1.1B (4-bit) | ~600 MB |
| Code | python_code_instructions_18k_alpaca | 300 | TinyLlama-1.1B (4-bit) | ~600 MB |
| Analysis | allenai/sciq | 300 | TinyLlama-1.1B (4-bit) | ~600 MB |

| Parameter | Value |
|-----------|-------|
| Method | QLoRA |
| LoRA rank | 16 |
| LoRA alpha | 32 |
| Epochs | 3 |
| Effective batch size | 8 |
| Learning rate | 2e-4 |
| Max sequence length | 1024 |
| Export format | Q4_K_M GGUF |
| Hardware | Colab T4 (16GB VRAM) |
| Total time | ~20–30 minutes |